In [2]:
%pip install elasticsearch

You should consider upgrading via the '/Users/akshaypatade/Desktop/Projects/purchase-orders/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [63]:
from elasticsearch import Elasticsearch, exceptions
from urllib.request import urlopen
import json
import time

In [64]:
client = Elasticsearch(hosts = "https://5856681e52d64c35a4b39da6e44c832b.us-central1.gcp.cloud.es.io:443", api_key = "Rzd0UGU1TUIwRFpHXzJVU2dYaHc6STE0c1NTUUpUbUdLYk1mMUJIb21yQQ==")

In [65]:
print(client.info())

{'name': 'instance-0000000000', 'cluster_name': '5856681e52d64c35a4b39da6e44c832b', 'cluster_uuid': 'rgYgR2nNRx6vydAPWjUj_w', 'version': {'number': '8.16.1', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': 'ffe992aa682c1968b5df375b5095b3a21f122bf3', 'build_date': '2024-11-19T16:00:31.793213192Z', 'build_snapshot': False, 'lucene_version': '9.12.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


In [52]:
try:
    client.inference.delete(inference_id="my-elser-endpoint")
except exceptions.NotFoundError:
    # Inference endpoint does not exist
    pass

try:
    client.options(
        request_timeout=60, max_retries=3, retry_on_timeout=True
    ).inference.put(
        task_type="sparse_embedding",
        inference_id="my-elser-endpoint",
        body={
            "service": "elser",
            "service_settings": {"num_allocations": 1, "num_threads": 1},
        },
    )
    print("Inference endpoint created successfully")
except exceptions.BadRequestError as e:
    if e.error == "resource_already_exists_exception":
        print("Inference endpoint created successfully")
    else:
        raise e

/var/folders/m8/9fkcwstj36jc40jlydh29f9r0000gn/T/ipykernel_14433/1915381603.py:2: GeneralAvailabilityWarning: This API is in technical preview and may be changed or removed in a future release. Elastic will work to fix any issues, but features in technical preview are not subject to the support SLA of official GA features.
  client.inference.delete(inference_id="my-elser-endpoint")
/var/folders/m8/9fkcwstj36jc40jlydh29f9r0000gn/T/ipykernel_14433/1915381603.py:8: GeneralAvailabilityWarning: This API is in technical preview and may be changed or removed in a future release. Elastic will work to fix any issues, but features in technical preview are not subject to the support SLA of official GA features.
  client.options(


Inference endpoint created successfully


/var/folders/m8/9fkcwstj36jc40jlydh29f9r0000gn/T/ipykernel_14433/1915381603.py:8: ElasticsearchWarning: Putting elasticsearch service inference endpoints (including elser service) without a model_id field is deprecated and will be removed in a future release. Please specify a model_id field.
  client.options(
/var/folders/m8/9fkcwstj36jc40jlydh29f9r0000gn/T/ipykernel_14433/1915381603.py:8: ElasticsearchWarning: The [elser] service is deprecated and will be removed in a future release. Use the [elasticsearch] service instead, with [model_id] set to [.elser_model_2_linux-x86_64] in the [service_settings]
  client.options(


In [53]:
inference_endpoint_info = client.inference.get(inference_id="my-elser-endpoint")
model_id = inference_endpoint_info["endpoints"][0]["service_settings"]["model_id"]

while True:
    status = client.ml.get_trained_models_stats(
        model_id=model_id,
    )

    deployment_stats = status["trained_model_stats"][0].get("deployment_stats")
    if deployment_stats is None:
        print("ELSER Model is currently being deployed.")
        time.sleep(5)
        continue

    nodes = deployment_stats.get("nodes")
    if nodes is not None and len(nodes) > 0:
        print("ELSER Model has been successfully deployed.")
        break
    else:
        print("ELSER Model is currently being deployed.")
    time.sleep(5)

ELSER Model has been successfully deployed.


/var/folders/m8/9fkcwstj36jc40jlydh29f9r0000gn/T/ipykernel_14433/734450827.py:1: GeneralAvailabilityWarning: This API is in technical preview and may be changed or removed in a future release. Elastic will work to fix any issues, but features in technical preview are not subject to the support SLA of official GA features.
  inference_endpoint_info = client.inference.get(inference_id="my-elser-endpoint")


In [61]:
client.indices.delete(index="semantic-product-search", ignore_unavailable=True)
client.indices.create(
    index="semantic-product-search",
    mappings={
        "properties": {
            "id": {"type": "text"},
            "type": {"type": "text"},
            "material" :{"type": "text"},
            "size" : {"type": "text"},
            "length" : {"type": "text"},
            "coating": {"type": "text"},
            "thread_type" : {"type": "text"},
            "description" : {"type": "text", "copy_to" : "product_search_semantic"},

            "product_search_semantic": {
                "type": "semantic_text",
                "inference_id": "my-elser-endpoint",
            },
        }
    },
)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'semantic-product-search'})

In [70]:
def pretty_search_response(response):
    if len(response["hits"]["hits"]) == 0:
        print("Your search returned no results.")
    else:
        for hit in response["hits"]["hits"]:
            id = hit["_id"]
            score = hit["_score"]
            type = hit["_source"]["type"]
            material = hit["_source"]["material"]
            size = hit["_source"]["size"]
            length = hit["_source"]["length"]
            coating = hit["_source"]["coating"]
            thread_type = hit["_source"]["thread_type"]
            description = hit["_source"]["description"]

            pretty_output = f"\nID: {id}\nScore: {score}\nType: {type}\nMaterial: {material}\nSize: {size}\nLength: {length}\nCoating: {coating}\nthread_type: {thread_type} \ndescription: {description} "

            print(pretty_output)

In [ ]:
#Once the data is populated in the elastic search, it is time to run the query and retrieve the elements
response = client.search(
    index="semantic-product-search",
    query={"semantic": {"field": "product_search_semantic", "query": "Steel Bolt 40mm"}},
)

pretty_search_response(response)


ID: f416fc32-852a-4617-9f03-2fed1713a1e7
Score: 12.982039
Type: Washer
Material: Stainless Steel
Size: M8
Length: 50mm
Coating: Black Oxide
thread_type: Fine 
description: Stainless Steel Washer M8 50mm Black Oxide Fine 

ID: e41144e8-7117-4ff1-8ca7-421de5056e9c
Score: 12.962876
Type: Washer
Material: Steel
Size: 1/2"
Length: 10mm
Coating: Black Oxide
thread_type: Machine 
description: Steel Washer 1/2" 10mm Black Oxide Machine 

ID: a5e24d83-8b71-4702-8dc5-f334c17a5f19
Score: 12.938309
Type: Washer
Material: Steel
Size: 3/4"
Length: 10mm
Coating: Black Oxide
thread_type: Machine 
description: Steel Washer 3/4" 10mm Black Oxide Machine 

ID: a603e206-62ad-408a-8359-1251b15ba76d
Score: 12.918321
Type: Washer
Material: Steel
Size: M8
Length: 100mm
Coating: Black Oxide
thread_type: Machine 
description: Steel Washer M8 100mm Black Oxide Machine 

ID: 94c24181-d59b-4307-b056-caa6f5c62c69
Score: 12.864363
Type: Washer
Material: Steel
Size: M8
Length: 30mm
Coating: Black Oxide
thread_type: 